# Per-gene / per-region colocboost across contexts (and optionally GWAS) in a single study

## Description

For a single study's `QtlDataset` RDS, run `pecotmr::colocboostPipeline()` per fan-out unit (gene or region). Three pipeline variants can be enabled independently:

- `--xqtl-coloc` (default ON; opt out with `--no-xqtl-coloc=true` or `xqtl_coloc=False` in SoS): within-study cross-context QTL coloc.
- `--joint-gwas` (default OFF): xQTL + GWAS joint coloc with the focal outcome from the GWAS. Requires `--gwas-sumstats`.
- `--separate-gwas` (default OFF): one focal-GWAS coloc per GWAS study. Requires `--gwas-sumstats`.

When `--gwas-sumstats` is supplied it must point at a QC'd `GwasSumStats` RDS (output of `summaryStatsQc()` upstream).

## Inputs

- `--qtl-dataset` &mdash; QtlDataset RDS produced by `qtl_dataset.ipynb`.
- `--genes ID1 ID2 …` **or** `--regions chr:start-end …` (mutually exclusive).
- `--cis-window` &mdash; bp window (gene mode default 1e6).
- `--gwas-sumstats` &mdash; QC'd GwasSumStats RDS path (required for joint/separate variants).
- `--xqtl-coloc` / `--joint-gwas` / `--separate-gwas` &mdash; pipeline-variant toggles.
- `--modular-script-dir` &mdash; directory containing the per-gene worker R scripts. Default `code/script`.

## Output

- `{cwd}/{study}.{gene|region}.colocboost.rds` per fan-out unit. The RDS holds the colocboostPipeline list with slots `xqtl_coloc`, `joint_gwas`, `separate_gwas`, `computing_time`; unused-variant slots are NULL.


## Example

```bash
# xQTL-only (default)
sos run pipeline/colocboost.ipynb colocboost \
    --cwd output --modular-script-dir /path/to/code/script \
    --study TEST_STUDY \
    --qtl-dataset output/TEST_STUDY.qtl_dataset.rds \
    --genes ENSG00000060237

# xQTL + joint-GWAS
sos run pipeline/colocboost.ipynb colocboost \
    --cwd output --modular-script-dir /path/to/code/script \
    --study TEST_STUDY \
    --qtl-dataset output/TEST_STUDY.qtl_dataset.rds \
    --gwas-sumstats output/GWAS.qcd.rds \
    --joint-gwas True \
    --genes ENSG00000060237
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: qtl_dataset = path
parameter: modular_script_dir = path('code/script')
# Mutually exclusive fan-out sources.
parameter: genes = []
parameter: regions = []
parameter: cis_window = 1000000
# GWAS-side input: a QC'd GwasSumStats RDS (build it upstream).
parameter: gwas_sumstats = path('.')
# Pipeline variants.
parameter: xqtl_coloc = True       # within-study cross-context (default ON)
parameter: joint_gwas = False      # xQTL + GWAS joint (requires gwas_sumstats)
parameter: separate_gwas = False   # per-GWAS-study separate-focal (requires gwas_sumstats)
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[colocboost]
if bool(genes) == bool(regions):
    raise ValueError("Specify exactly one of --genes (trait IDs) or --regions (chr:start-end strings).")
if (joint_gwas or separate_gwas) and not gwas_sumstats.is_file():
    raise ValueError("--joint-gwas / --separate-gwas require --gwas-sumstats pointing at a QC'd GwasSumStats RDS.")
if not (xqtl_coloc or joint_gwas or separate_gwas):
    raise ValueError("Enable at least one of xqtl_coloc, joint_gwas, separate_gwas.")
fanout_items = genes if genes else regions
fanout_kind  = 'gene' if genes else 'region'
def _fanout_safe(s):
    return s.replace(':', '_').replace('-', '_')
input: qtl_dataset, for_each = 'fanout_items'
output: f"{cwd}/{study}.{_fanout_safe(_fanout_items)}.colocboost.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/colocboost.R \
        --qtl-dataset ${_input} \
        ${('--gene-id ' + _fanout_items + ' --cis-window ' + str(cis_window)) if fanout_kind == 'gene' else ('--region ' + _fanout_items)} \
        ${'--gwas-sumstats ' + str(gwas_sumstats) if gwas_sumstats.is_file() else ''} \
        ${'--no-xqtl-coloc' if not xqtl_coloc else ''} \
        ${'--joint-gwas' if joint_gwas else ''} \
        ${'--separate-gwas' if separate_gwas else ''} \
        --output ${_output}
